In [ ]:
!pip install pyspark

In [ ]:
# 1. Imports & Spark session
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, regexp_replace, length
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pathlib, os

In [ ]:
spark = SparkSession.builder.appName("Lyrics7Genres").getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.0.2


In [ ]:
# 2. Paths & config
MEND_CSV  = "/content/Merged_dataset.csv"
TRAIN_DIR = "data/train80"
TEST_DIR  = "data/test20"
SEED      = 42
os.makedirs("data", exist_ok=True)


In [ ]:
# 3. Load & preview dataset
# Reads the CSV into a Spark DataFrame.
df = (
    spark.read
         .option("header", "true")
         .option("multiLine", "true")
         .csv(MEND_CSV)
)
df.show(5, truncate=80)


+--------------------+--------------------+------------+-----+--------------------------------------------------------------------------------+
|         artist_name|          track_name|release_date|genre|                                                                          lyrics|
+--------------------+--------------------+------------+-----+--------------------------------------------------------------------------------+
|              mukesh|mohabbat bhi jhoothi|        1950|  pop|hold time feel break feel untrue convince speak voice tear try hold hurt try ...|
|       frankie laine|           i believe|        1950|  pop|believe drop rain fall grow believe darkest night candle glow believe go astr...|
|         johnnie ray|                 cry|        1950|  pop|sweetheart send letter goodbye secret feel better wake dream think real false...|
|         pérez prado|            patricia|        1950|  pop|kiss lips want stroll charm mambo chacha meringue heaven arm japan brag ge

In [ ]:
# 4. Minimal cleaning
df = (
    df.select("artist_name", "track_name", "release_date", "genre", "lyrics")  #Keeps only the five columns required in the assignment.
      .withColumn("genre", trim(lower(col("genre"))))
      .withColumn("lyrics", regexp_replace(col("lyrics"), r"\s+", " "))
      .filter(length(col("lyrics")) > 0)
      .filter(col("release_date").rlike(r"^\d{4}$"))     # keep 4-digit year
)
print("Clean rows:", df.count())

Clean rows: 28471


In [ ]:
# 5. 80 / 20 train-test split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=SEED)
train_df.write.mode("overwrite").parquet(TRAIN_DIR)
test_df.write.mode("overwrite").parquet(TEST_DIR)
print("Train:", train_df.count(), " Test:", test_df.count())

Train: 22803  Test: 5668


In [ ]:
# 6. Build feature pipeline  (same as before, up to IDF) -----------------------
tokenizer = RegexTokenizer(inputCol="lyrics", outputCol="tokens", pattern="\\W")
stop_rm   = StopWordsRemover(inputCol="tokens", outputCol="nostop")
tf        = HashingTF(inputCol="nostop", outputCol="tf", numFeatures=1 << 18)
idf       = IDF(inputCol="tf", outputCol="features")
lab       = StringIndexer(inputCol="genre", outputCol="label")

# 6b. Classifier + param grid --------------------------------------------------
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

lr = LogisticRegression(
        maxIter=100,
        elasticNetParam=1.0,
        featuresCol="features",
        labelCol="label"
)

paramGrid = (
    ParamGridBuilder()
      .addGrid(lr.regParam, [0.01, 0.05, 0.1, 0.3])
      .build()
)

evaluator = MulticlassClassificationEvaluator(metricName="f1")

tvs = TrainValidationSplit(
        estimator=lr,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        trainRatio=0.8,           # 80 % of train_df used for fitting, 20 % FOR VALIDATION
        seed=SEED
)

pipe = Pipeline(stages=[tokenizer, stop_rm, tf, idf, lab, tvs])


In [ ]:
# 7. Train model (unchanged call, but now includes tuning)
model = pipe.fit(train_df)

best_lr = model.stages[-1].bestModel
print("Best regParam:", best_lr._java_obj.getRegParam())

Best regParam: 0.01


In [ ]:
# 8. Evaluate on the held-out test set
preds = model.transform(test_df)
for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    val = evaluator.setMetricName(m).evaluate(preds)
    print(f"{m:>18}: {val:.4f}")

          accuracy: 0.3199
                f1: 0.2535
 weightedPrecision: 0.5001
    weightedRecall: 0.3199


In [ ]:
import os, sys
# absolute path of the conda-env python that Jupyter is using now
PY = sys.executable

os.environ["PYSPARK_PYTHON"]        = PY   # workers
os.environ["PYSPARK_DRIVER_PYTHON"] = PY   # driver

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("Lyrics7Genres")
         .config("spark.pyspark.python", PY)
         .config("spark.pyspark.driver.python", PY)
         .getOrCreate())

print("Driver python :", sys.executable)
print("Worker python :", spark.sparkContext.pythonExec)   # should match to save the model


Driver python : /usr/bin/python3
Worker python : python3


In [ ]:
import pathlib, shutil

# 1️⃣  Path:  `<notebook directory>/model_stage2`
save_dir = pathlib.Path.cwd() / "model_stage4_merged_Trans_way_new_merged"

# 2️⃣  Remove any previous run so .overwrite() won’t clash with a *file*
if save_dir.exists():
    shutil.rmtree(save_dir)

# 3️⃣  Persist the fitted pipeline
# Spark is happy with either a plain absolute path or a file:// URI.
# We'll use the plain path to keep it readable.
model.write().overwrite().save(str(save_dir))

print("✅  Model saved to:", save_dir.resolve())


✅  Model saved to: /content/model_stage4_merged_Trans_way_new_merged


In [ ]:
from pyspark.ml import PipelineModel #CHECK
reloaded = PipelineModel.load(f"model_stage4_merged_Trans_way_new_merged")
print("Reload OK, stages:", len(reloaded.stages))


Reload OK, stages: 6


In [ ]:
!zip -r model_merged.zip model_stage4_merged_Trans_way_new_merged/

  adding: model_stage4_merged_Trans_way_new_merged/ (stored 0%)
  adding: model_stage4_merged_Trans_way_new_merged/stages/ (stored 0%)
  adding: model_stage4_merged_Trans_way_new_merged/stages/3_IDF_5ff225ea4946/ (stored 0%)
  adding: model_stage4_merged_Trans_way_new_merged/stages/3_IDF_5ff225ea4946/data/ (stored 0%)
  adding: model_stage4_merged_Trans_way_new_merged/stages/3_IDF_5ff225ea4946/data/part-00000-c0942f33-0d34-4075-98b8-bf96832a6875-c000.snappy.parquet (deflated 14%)
  adding: model_stage4_merged_Trans_way_new_merged/stages/3_IDF_5ff225ea4946/data/._SUCCESS.crc (stored 0%)
  adding: model_stage4_merged_Trans_way_new_merged/stages/3_IDF_5ff225ea4946/data/.part-00000-c0942f33-0d34-4075-98b8-bf96832a6875-c000.snappy.parquet.crc (stored 0%)
  adding: model_stage4_merged_Trans_way_new_merged/stages/3_IDF_5ff225ea4946/data/_SUCCESS (stored 0%)
  adding: model_stage4_merged_Trans_way_new_merged/stages/3_IDF_5ff225ea4946/metadata/ (stored 0%)
  adding: model_stage4_merged_Trans_wa